# 1D CNN — cfDNA Cancer Classifier

**Why a CNN on cfDNA data:**  
RF, XGBoost, and MLP all used 18 features (13 bins + 5 scalars). The bins aggregate 10 bp windows — useful, but they discard shape information within each window.

The CNN uses the raw 1 bp histogram: `hist_090` through `hist_400` — 311 values representing the full fragment length distribution at single-base resolution. This is the same signal a clinician would look at when reading a DELFI plot.

A 1D CNN treats this 311-point curve as a signal. Each convolutional filter slides along the length axis learning to detect local patterns — a dip at 167 bp (nucleosome peak), elevated density at 100–150 bp (sub-nucleosomal), reduced long fragments. These spatial patterns are exactly what distinguishes cancer from healthy and cannot be captured by a single scalar.

**Architecture:**  
Two conv layers → max-pool → flatten → dropout → FC. Small filters, heavy regularisation (dropout 0.5) to prevent overfitting on 148 samples.

**Framework: PyTorch**  
More control over the training loop — useful for seeing exactly what's happening with loss, early stopping, and class weighting on a small dataset.

**Data:**  
Different from previous notebooks — we load `features_model.csv` directly and extract the 311 hist columns. The same 80/20 stratified split (random_state=42) is reproduced to guarantee identical train/test partitions.

## Step 1 — Load histogram features + reproduce split

In [ ]:
import numpy as np
import pandas as pd
import pathlib, pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

ROOT = pathlib.Path().resolve().parent

df = pd.read_csv(ROOT / 'data' / 'processed' / 'features_model.csv')

# 311 histogram features: hist_090 … hist_400
hist_cols = [c for c in df.columns if c.startswith('hist_')]
hist_cols.sort()

X_hist = df[hist_cols].astype(float).values
y      = (df['label'] == 'cancer').astype(int).values

# Reproduce the exact same 80/20 split as notebook 01 (same seed, same stratify)
X_train_h, X_test_h, y_train, y_test = train_test_split(
    X_hist, y, test_size=0.20, stratify=y, random_state=42
)

# Fit scaler on train hist features only
scaler_hist = StandardScaler()
X_train_h = scaler_hist.fit_transform(X_train_h)
X_test_h  = scaler_hist.transform(X_test_h)

print(f'Histogram features : {len(hist_cols)}  ({hist_cols[0]} … {hist_cols[-1]})')
print(f'X_train : {X_train_h.shape}  (healthy={(y_train==0).sum()}  cancer={(y_train==1).sum()})')
print(f'X_test  : {X_test_h.shape}   (healthy={(y_test==0).sum()}   cancer={(y_test==1).sum()})')

## Step 2 — Define the CNN architecture

**Signal flow through the network:**

```
Input       (batch, 1, 311)   ← 311-point histogram, 1 channel
Conv1 k=15  (batch, 8, 311)   ← 8 filters, each 15 bp wide
MaxPool /4  (batch, 8,  77)   ← compress 4× — keeps dominant peaks
Conv2 k=7   (batch, 16,  77)  ← 16 filters, each 7 bp wide
MaxPool /4  (batch, 16,  19)  ← compress again
Flatten     (batch, 304)
Dropout 0.5                   ← randomly zero half the neurons each step
Linear→32   (batch, 32)
Dropout 0.3
Linear→1    (batch, 1)        ← raw logit (no sigmoid — handled by loss)
```

**Why large kernels (15, 7):**  
A 15 bp kernel spans a biologically meaningful window — the width of a short cfDNA fragment. The filter can learn "elevated density around 130–145 bp" as a single feature. A 3 bp kernel would be too narrow to detect these broad peaks.

**Why `BCEWithLogitsLoss` instead of sigmoid + BCE:**  
Numerically more stable — combines sigmoid and binary cross-entropy in one operation using the log-sum-exp trick. Avoids gradient saturation near 0 and 1.

**Why `pos_weight`:**  
The loss function's equivalent of `class_weight='balanced'`. Set to healthy/cancer = 95/53 ≈ 1.79 — each cancer sample's loss contribution is upweighted 1.79×.

**Why batch size = 32 on 148 samples:**  
Too small (e.g. 8) → noisy gradient updates. Too large (all 148) → no stochasticity, slower generalisation. 32 is the standard starting point.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

print(f'PyTorch version : {torch.__version__}')
print(f'Device          : {"cuda" if torch.cuda.is_available() else "cpu"}')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


class CfDNA_CNN(nn.Module):
    def __init__(self, dropout1=0.5, dropout2=0.3):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(1, 8,  kernel_size=15, padding=7),  # → (B, 8, 311)
            nn.ReLU(),
            nn.MaxPool1d(4),                               # → (B, 8,  77)
            nn.Conv1d(8, 16, kernel_size=7,  padding=3),  # → (B,16,  77)
            nn.ReLU(),
            nn.MaxPool1d(4),                               # → (B,16,  19)
        )
        self.fc = nn.Sequential(
            nn.Flatten(),                                  # → (B, 304)
            nn.Dropout(dropout1),
            nn.Linear(304, 32),
            nn.ReLU(),
            nn.Dropout(dropout2),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        return self.fc(self.conv(x)).squeeze(1)


# Verify shapes with a dummy batch
dummy = torch.zeros(4, 1, 311)
model_test = CfDNA_CNN()
out = model_test(dummy)
print(f'\nDummy forward pass — input {tuple(dummy.shape)} → output {tuple(out.shape)}')

total_params = sum(p.numel() for p in model_test.parameters() if p.requires_grad)
print(f'Trainable parameters : {total_params:,}')

## Step 3 — 5-fold CV training

**Training loop design:**

- `pos_weight = 95/53 ≈ 1.79` — passed to `BCEWithLogitsLoss` to upweight cancer samples
- Adam optimiser, `lr=1e-3`, weight decay = 0 (dropout handles regularisation)
- Batch size = 32
- Max 200 epochs, early stopping on **validation AUC** with patience = 20

**Why stop on AUC not loss:**  
Loss can keep decreasing while AUC plateaus or worsens — especially with class imbalance. Stopping on AUC directly optimises the metric we care about.

**Why reinitialise the model each fold:**  
Each fold must start from the same random state to isolate the effect of the data partition. Reusing weights from a previous fold would leak information across folds.

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

BATCH_SIZE = 32
MAX_EPOCHS = 200
PATIENCE   = 20
LR         = 1e-3

pos_weight = torch.tensor([(y_train == 0).sum() / (y_train == 1).sum()],
                           dtype=torch.float32).to(DEVICE)
print(f'pos_weight = {pos_weight.item():.4f}')


def train_cnn_fold(X_tr, y_tr, X_val, y_val, seed=42):
    torch.manual_seed(seed)
    model = CfDNA_CNN().to(DEVICE)
    opt   = torch.optim.Adam(model.parameters(), lr=LR)
    loss_fn = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor([(y_tr == 0).sum() / (y_tr == 1).sum()],
                                dtype=torch.float32).to(DEVICE)
    )

    # DataLoader for training
    Xt = torch.tensor(X_tr[:, None, :], dtype=torch.float32).to(DEVICE)  # (N,1,311)
    yt = torch.tensor(y_tr,             dtype=torch.float32).to(DEVICE)
    Xv = torch.tensor(X_val[:, None, :], dtype=torch.float32).to(DEVICE)
    yv = y_val

    loader = DataLoader(TensorDataset(Xt, yt), batch_size=BATCH_SIZE, shuffle=True)

    best_auc, best_epoch, patience_count = 0.0, 0, 0

    for epoch in range(MAX_EPOCHS):
        model.train()
        for xb, yb in loader:
            opt.zero_grad()
            loss_fn(model(xb), yb).backward()
            opt.step()

        # Validation AUC
        model.eval()
        with torch.no_grad():
            logits = model(Xv).cpu().numpy()
        proba = torch.sigmoid(torch.tensor(logits)).numpy()
        auc   = roc_auc_score(yv, proba)

        if auc > best_auc:
            best_auc, best_epoch = auc, epoch
            patience_count = 0
        else:
            patience_count += 1
            if patience_count >= PATIENCE:
                break

    return best_auc, best_epoch


cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_aucs, fold_epochs = [], []

for fold, (tr_idx, val_idx) in enumerate(cv.split(X_train_h, y_train)):
    auc, ep = train_cnn_fold(
        X_train_h[tr_idx], y_train[tr_idx],
        X_train_h[val_idx], y_train[val_idx],
        seed=42 + fold,
    )
    fold_aucs.append(auc)
    fold_epochs.append(ep)
    print(f'  Fold {fold+1}  AUC={auc:.4f}  best_epoch={ep}')

scores = np.array(fold_aucs)
print()
print('CNN — 5-fold CV AUC')
print(f'  Fold scores : {[round(s, 4) for s in scores]}')
print(f'  Mean AUC    : {scores.mean():.4f}')
print(f'  Std         : {scores.std():.4f}')
print(f'  Range       : {scores.min():.4f} – {scores.max():.4f}')
print()
print(f'RF  (tuned) : 0.8389')
print(f'MLP (tuned) : 0.8214')
print(f'XGB (tuned) : 0.8113')
print(f'CNN (base)  : {scores.mean():.4f}')

## Step 4 — Final model fit + Save

**Why the CNN wins by such a large margin:**  
The tabular models used 18 features — aggregated bins and scalars. The CNN gets the full 311-point histogram at 1 bp resolution. The difference is that convolutions detect *spatial patterns*: the dip at the 167 bp nucleosome peak, the enrichment at 100–150 bp, the depletion of long fragments. These are correlated across adjacent base pairs — a 15 bp convolutional filter naturally captures that correlation. A tree or MLP treating each bp independently cannot.

**Final training strategy:**  
Refit on all 148 samples. Since we can't use a held-out set for early stopping now (we used CV to pick the architecture), we train for a fixed number of epochs. We use the median best_epoch across the 5 folds as a proxy for how many epochs the model needs.

In [ ]:
import math

# Fold best epochs are bimodal: [10, 81, 14, 10, 81]
# Median (14) is too few — 2/5 folds needed 81 epochs to converge.
# Use max + a small buffer so the final model fully converges.
n_epochs_final = max(fold_epochs) + 20
print(f'Fold best epochs : {fold_epochs}')
print(f'Training final model for {n_epochs_final} epochs on all 148 samples')
print()

torch.manual_seed(42)
cnn_final = CfDNA_CNN().to(DEVICE)
opt_final = torch.optim.Adam(cnn_final.parameters(), lr=LR)
loss_fn_final = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor([(y_train == 0).sum() / (y_train == 1).sum()],
                            dtype=torch.float32).to(DEVICE)
)

X_all = torch.tensor(X_train_h[:, None, :], dtype=torch.float32).to(DEVICE)
y_all = torch.tensor(y_train,               dtype=torch.float32).to(DEVICE)
loader_all = DataLoader(TensorDataset(X_all, y_all), batch_size=BATCH_SIZE, shuffle=True)

for epoch in range(n_epochs_final):
    cnn_final.train()
    for xb, yb in loader_all:
        opt_final.zero_grad()
        loss_fn_final(cnn_final(xb), yb).backward()
        opt_final.step()

# Quick sanity check: training AUC
cnn_final.eval()
with torch.no_grad():
    train_logits = cnn_final(X_all).cpu().numpy()
train_proba = torch.sigmoid(torch.tensor(train_logits)).numpy()
train_auc   = roc_auc_score(y_train, train_proba)

# Save model weights
torch.save(cnn_final.state_dict(), ROOT / 'models' / 'cnn_final.pt')

# Also save the scaler (needed for inference)
with open(ROOT / 'models' / 'scaler_hist.pkl', 'wb') as f:
    pickle.dump(scaler_hist, f)

print(f'Saved: models/cnn_final.pt')
print(f'Saved: models/scaler_hist.pkl')
print(f'Training AUC (sanity check) : {train_auc:.4f}  (expect ~0.95+)')
print()
print('=' * 50)
print('ALL MODELS — CV AUC COMPARISON')
print('=' * 50)
print(f'  CNN           (base)  : 0.8990')
print(f'  Random Forest (tuned) : 0.8389')
print(f'  MLP           (tuned) : 0.8214')
print(f'  XGBoost       (tuned) : 0.8113')
print('=' * 50)
print()
print('Test set is still locked. Final AUC reported in evaluation notebook.')